In [2]:
import pandas as pd

interactions = pd.read_parquet("data\\interactions.parquet")

interactions.head(10)

,user_idx,movie_idx,rating
0,0,1104,5.0
1,0,639,3.0
2,0,853,3.0
3,0,3177,4.0
4,0,2162,5.0
5,0,1107,3.0
6,0,1195,5.0
7,0,2599,5.0
8,0,580,4.0
9,0,858,4.0


In [7]:
def recommend_popular(
        user_idx: int,
        interactions: pd.DataFrame,
        k: int = 10
):
    popularity = (
        interactions.groupby("movie_idx").rating
        .agg(
            rating_count="count",
            avg_rating="mean"
        )
        .reset_index()
        .sort_values(by=["rating_count", "avg_rating"], ascending=False)
    )

    watched_movies = set(
        interactions.loc[
            interactions.user_idx == user_idx, "movie_idx"
        ]
    )

    

    return (
        popularity[
            ~popularity.movie_idx.isin(watched_movies)
        ]
        .head(k)
    )

print(recommend_popular(user_idx=3, interactions=interactions, k=10))


      movie_idx  rating_count  avg_rating
2651       2651          3428    4.317386
575         575          2649    4.058513
2374       2374          2590    4.315830
1178       1178          2583    3.990321
579         579          2578    4.351823
1449       1449          2538    3.739953
593         593          2513    4.254676
2557       2557          2459    4.406263
106         106          2443    4.234957
2203       2203          2369    4.127480


In [6]:
def bayesian_popularity(
    interactions: pd.DataFrame,
    m: int = 100
):
    C = interactions.rating.mean()

    popularity = (
        interactions.groupby("movie_idx").rating
        .agg(
            rating_count="count",
            avg_rating="mean"
        )
        .reset_index()
    )

    popularity["score"] = (
        popularity.rating_count 
        / (popularity.rating_count + m) 
        * popularity.avg_rating 
        + m 
        / (popularity.rating_count + m)
        * C
    )

    return popularity.sort_values(by="score", ascending=False)

def recommend_bayesian_popularity(
    user_idx: int,
    popularity: pd.DataFrame,
    interactions: pd.DataFrame,
    k: int = 10
):
    
    watched_movies = set(
        interactions.loc[
            interactions.user_idx == user_idx, "movie_idx"
        ]
    )

    return (
        popularity[
            ~popularity.movie_idx.isin(watched_movies)
        ]
        .head(k)
    )


model = bayesian_popularity(interactions=interactions, m=100)

print(recommend_bayesian_popularity(3, model, interactions, k=10))

      movie_idx  rating_count  avg_rating     score
309         309          2227    4.554558  4.512745
802         802          2223    4.524966  4.484355
513         513          2304    4.510417  4.471779
49           49          1783    4.517106  4.467422
1839       1839           628    4.560510  4.426039
1066       1066           882    4.507936  4.413601
843         843          1050    4.476191  4.398397
708         708           657    4.520548  4.396508
713         713          1367    4.449890  4.390700
2557       2557          2459    4.406263  4.374036
